# Day 15 · Zillacode prep 4/4: hard patterns, Spark SQL, debugging and a mock exam

The last prep day:
1. Four patterns behind Zillacode's Hard problems: sessions, streaks, first and last row per group, the latest version from several snapshots, plus comparing with the whole table
2. Spark SQL inside `etl`
3. Debugging: the error messages you will meet, and the mistakes that give no error at all
4. UDFs: when you need one and when you don't
5. **Mock exam**: 8 problems in 90 minutes on new data, built exactly the way Zillacode builds its inputs

After this notebook, set up Zillacode (the last task of Day 15 in the roadmap) and start solving its 52 problems.

## How to work
* Run the **Setup** cell first, then the cell that prints the tables.
* **Lessons** explain one tool at a time. Read them, run every example cell (the output is already shown under it), then change the example and run it again until you can predict the result.
* **Exercises** come right after each lesson and are in Zillacode's format: complete `etl(...)` in the cell under the task so that it **returns** a DataFrame, then run the cell. The last line, `etl(...).show()`, prints your result. It must match the expected output: same columns, same rows, same order. (Zillacode itself ignores the order of rows and columns; the expected outputs here are sorted only so you can compare them by eye.)
* The answers are collapsed at the very end of the notebook. Open one only after you have tried, then close it and write the solution again yourself.
* Today: about 1 hour of lessons, 45 minutes of exercises (5), then a 90-minute mock exam (8 problems).

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook.

## The data
A small online shop. The setup cell creates every table from rows you can read, and the cell after it prints them.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **signups_raw**: sign-ups exactly as a web form sent them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text in three formats), *amount* (text), *prefs* (JSON text)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; review 6 has no tags; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

## Setup
Run this cell once. It starts Spark and creates the tables.

In [ ]:
# Setup: starts Spark and creates the tables. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")
         .appName("zillacode-prep-15")
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.shuffle.partitions", "4")      # small data: 4 shuffle partitions are enough
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags (text)
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ",    "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                     "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow",  "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",          "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",               "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",                None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",                  "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",         "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text, three formats), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

print("Spark", spark.version, "is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, "
      "promotions, events, referrals")

Spark 4.2.0 is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals


Print every table, so you can check results by eye:

In [ ]:
for name in ["customers", "products", "orders", "reviews", "signups_raw", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 1 · Four patterns behind the Hard problems

Day 14 already covered top N per group, the latest row per key, running totals and growth. The Hard problems mostly add one of these four ideas.

### A. Sessions: split rows into groups wherever there's a gap
"A new session starts when more than 30 minutes passed since the previous event." The recipe:
1. Compute the gap to the previous row with `lag` (per user, ordered by time).
2. Make a flag column: 1 where a new group starts (the gap is too big, or there's no previous row), else 0.
3. A **running sum of the flag** numbers the groups: 1, 1, 1, 2, 2, 3...

```python
w = W.partitionBy("user").orderBy("time")
gap = F.unix_timestamp("time") - F.unix_timestamp(F.lag("time").over(w))           # seconds
flag = F.when(gap.isNull() | (gap > 30 * 60), 1).otherwise(0)
df.withColumn("new", flag).withColumn("session", F.sum("new").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)))
```
After that, `groupBy(user, session)` gives each session's start, end, length and number of events.

In [ ]:
# Shopping trips: a new trip starts when more than 30 days passed since the customer's previous order
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
running = w.rowsBetween(W.unboundedPreceding, W.currentRow)
(orders.filter(F.col("customer_id").isin(1, 3))
       .withColumn("gap_days", F.datediff("order_date", F.lag("order_date").over(w)))
       .withColumn("new_trip", F.when(F.col("gap_days").isNull() | (F.col("gap_days") > 30), 1).otherwise(0))
       .withColumn("trip", F.sum("new_trip").over(running))
       .select("customer_id", "order_id", "order_date", "gap_days", "new_trip", "trip")
       .orderBy("customer_id", "order_date", "order_id")
       .show())

+-----------+--------+----------+--------+--------+----+
|customer_id|order_id|order_date|gap_days|new_trip|trip|
+-----------+--------+----------+--------+--------+----+
|          1|    1001|2024-01-10|    NULL|       1|   1|
|          1|    1002|2024-01-10|       0|       0|   1|
|          1|    1007|2024-02-20|      41|       1|   2|
|          1|    1015|2024-03-25|      34|       1|   3|
|          3|    1004|2024-02-03|    NULL|       1|   1|
|          3|    1009|2024-03-05|      31|       1|   2|
|          3|    1016|2024-03-28|      23|       0|   2|
+-----------+--------+----------+--------+--------+----+


### B. Streaks: runs of consecutive days ("gaps and islands")
"The longest run of consecutive days with a login." The trick: number each user's **distinct** days with `row_number`, then subtract that number from the date. Inside a run of consecutive days, the date goes up by 1 and the row number goes up by 1, so `day - row_number` stays **the same**. It changes as soon as a day is skipped. That constant (the "anchor") is a group key:

| day | row_number | day − row_number |
|---|---|---|
| 03-01 | 1 | 02-29 |
| 03-02 | 2 | 02-29 |
| 03-03 | 3 | 02-29 |
| 03-06 | 4 | 03-02 |
| 03-07 | 5 | 03-02 |

Then `groupBy(user, anchor)` gives each streak: `count` is its length, `min` and `max` its first and last day. Take the days `distinct` first: two logins on one day would break the arithmetic.

In [ ]:
logins = (spark.createDataFrame(
              [("ann", "2024-03-01"), ("ann", "2024-03-02"), ("ann", "2024-03-02"), ("ann", "2024-03-03"),
               ("ann", "2024-03-06"), ("ann", "2024-03-07"), ("bob", "2024-03-05")], "user string, day string")
          .withColumn("day", F.to_date("day"))
          .distinct())
w = W.partitionBy("user").orderBy("day")
islands = logins.withColumn("rn", F.row_number().over(w)).withColumn("anchor", F.date_sub("day", F.col("rn")))
islands.orderBy("user", "day").show()
(islands.groupBy("user", "anchor")
        .agg(F.min("day").alias("start"), F.max("day").alias("end"), F.count("*").alias("days"))
        .orderBy("user", "start")
        .show())

+----+----------+---+----------+
|user|       day| rn|    anchor|
+----+----------+---+----------+
| ann|2024-03-01|  1|2024-02-29|
| ann|2024-03-02|  2|2024-02-29|
| ann|2024-03-03|  3|2024-02-29|
| ann|2024-03-06|  4|2024-03-02|
| ann|2024-03-07|  5|2024-03-02|
| bob|2024-03-05|  1|2024-03-04|
+----+----------+---+----------+

+----+----------+----------+----------+----+
|user|    anchor|     start|       end|days|
+----+----------+----------+----------+----+
| ann|2024-02-29|2024-03-01|2024-03-03|   3|
| ann|2024-03-02|2024-03-06|2024-03-07|   2|
| bob|2024-03-04|2024-03-05|2024-03-05|   1|
+----+----------+----------+----------+----+


### C. The first and the last row of each group, in one `groupBy`
`F.min_by(value, ordering)` returns `value` from the row with the smallest `ordering`, and `F.max_by` from the row with the largest. Both work inside a normal `agg`, next to other aggregates. To break ties, order by a **struct**: `F.struct("order_date", "order_id")` compares the date first and the id second. The alternative is two `row_number` windows (one ascending, one descending) and a join, which is longer.

In [ ]:
by_time = F.struct("order_date", "order_id")
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.count("*").alias("orders"),
            F.min_by("order_id", by_time).alias("first_order"),
            F.max_by("order_id", by_time).alias("last_order"),
            F.max_by("status", by_time).alias("last_status"))
       .orderBy("customer_id")
       .show())

+-----------+------+-----------+----------+-----------+
|customer_id|orders|first_order|last_order|last_status|
+-----------+------+-----------+----------+-----------+
|          1|     4|       1001|      1015|    shipped|
|          2|     3|       1003|      1013|  delivered|
|          3|     3|       1004|      1016|  delivered|
|          4|     2|       1006|      1010|  cancelled|
|          5|     2|       1008|      1014|  delivered|
|          6|     1|       1012|      1012|    shipped|
+-----------+------+-----------+----------+-----------+


### D. The latest version of every record from several snapshots
Two (or more) copies of a table taken at different times, and you need the current state, including records that exist in only one copy:
1. Tag each copy with its date: `withColumn("snapshot", F.lit(...))`.
2. `unionByName` them.
3. `row_number` over `partitionBy(key).orderBy(F.desc("snapshot"))`, then keep 1.

The same recipe handles a single table with repeated keys and an "updated_at" column (Zillacode #30 removes duplicated keys this way).

In [ ]:
prices_jan = spark.createDataFrame([(101, 1250.0), (102, 150.0), (103, 90.0)], "product_id int, price double")
prices_feb = spark.createDataFrame([(101, 1200.0), (103, 85.0), (104, 60.0)], "product_id int, price double")
history = (prices_jan.withColumn("as_of", F.to_date(F.lit("2024-01-31")))
                     .unionByName(prices_feb.withColumn("as_of", F.to_date(F.lit("2024-02-29")))))
w = W.partitionBy("product_id").orderBy(F.desc("as_of"))
history.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn").orderBy("product_id").show()

+----------+------+----------+
|product_id| price|     as_of|
+----------+------+----------+
|       101|1200.0|2024-02-29|
|       102| 150.0|2024-01-31|
|       103|  85.0|2024-02-29|
|       104|  60.0|2024-02-29|
+----------+------+----------+


### E. Comparing every row with the whole table: three ways
"Products more expensive than the average", "rides whose rating is far from the overall average" (#32). The overall value can come from:
1. **A Python value**: `avg = df.agg(F.avg("x")).first()[0]`, then `F.lit(avg)`. Short, but it runs a separate job.
2. **A cross join with a one-row aggregate**: `df.crossJoin(df.agg(F.avg("x").alias("avg_x")))`. Stays in one plan.
3. **A window over the whole table**: `F.avg("x").over(W.partitionBy())`. No join, but every row goes to one partition.

All three give the same answer; on Zillacode's small inputs, choose whichever reads best.

In [ ]:
avg_price = products.agg(F.avg("price")).first()[0]
one = products.select("product_name", "price", F.lit(avg_price).alias("avg_all"))
two = products.crossJoin(products.agg(F.avg("price").alias("avg_all"))).select("product_name", "price", "avg_all")
three = products.select("product_name", "price", F.avg("price").over(W.partitionBy()).alias("avg_all"))
three.orderBy("product_name").show()
print("same result all three ways:", sorted(one.collect()) == sorted(two.collect()) == sorted(three.collect()))

+------------+------+-------+
|product_name| price|avg_all|
+------------+------+-------+
|     Blender|  60.0|  225.0|
|Coffee Maker|  85.0|  225.0|
|    Cookbook|  25.0|  225.0|
|   Desk Lamp|  40.0|  225.0|
|  Headphones| 150.0|  225.0|
|      Laptop|1200.0|  225.0|
|       Novel|  15.0|  225.0|
+------------+------+-------+

same result all three ways: True


### Exercises

**Q1 · `etl(events)`** Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than 30 minutes passed since the customer's previous event. Show customer_id, event_time, page and session (1, 2, ... per customer), sorted by customer_id, then event_time.

Expected output:

```
+-----------+-------------------+-------+-------+
|customer_id|         event_time|   page|session|
+-----------+-------------------+-------+-------+
|          1|2024-03-01 09:00:00|   home|      1|
|          1|2024-03-01 09:10:00|product|      1|
|          1|2024-03-01 11:00:00|   home|      2|
|          1|2024-03-02 10:00:00|   cart|      3|
|          1|2024-03-03 10:30:00|   home|      4|
|          1|2024-03-05 08:00:00|   home|      5|
|          2|2024-03-01 20:00:00|   home|      1|
|          2|2024-03-01 20:20:00|product|      1|
|          2|2024-03-01 20:45:00|   cart|      1|
|          2|2024-03-04 18:00:00|   home|      2|
|          2|2024-03-05 18:00:00|   home|      3|
+-----------+-------------------+-------+-------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

**Q2 · `etl(events)`** What is each customer's longest streak of consecutive days with at least one event? Show customer_id and longest_streak, sorted by customer_id.

Expected output:

```
+-----------+--------------+
|customer_id|longest_streak|
+-----------+--------------+
|          1|             3|
|          2|             2|
|          3|             3|
+-----------+--------------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

**Q3 · `etl(orders)`** Find the customers whose first order (by date, then order_id) was delivered but whose latest order was not. Columns: customer_id, first_status, last_status; sorted by customer_id.

Expected output:

```
+-----------+------------+-----------+
|customer_id|first_status|last_status|
+-----------+------------+-----------+
|          1|   delivered|    shipped|
+-----------+------------+-----------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q4 · `etl(customers, customers_new)`** Build the current customer list from two snapshots: `customers` (taken on 2024-03-31) and `customers_new` (taken on 2024-04-07). Keep the newest row of every customer, including customers found only in the older snapshot. Show customer_id, name, city and snapshot (the date of that row's snapshot), sorted by customer_id.

Expected output:

```
+-----------+----------------+--------+----------+
|customer_id|            name|    city|  snapshot|
+-----------+----------------+--------+----------+
|          1|    Aziz Karimov|Tashkent|2024-04-07|
|          2|   Bella Schmidt|  Munich|2024-04-07|
|          3|     Carlos Ruiz|  Madrid|2024-04-07|
|          4|Dilnoza Rahimova|Tashkent|2024-04-07|
|          5|     Emma Wilson|  London|2024-04-07|
|          6|    Farid Aliyev|    NULL|2024-04-07|
|          7|       Grace Lee|  Berlin|2024-03-31|
|          8|       Hana Sato|   Osaka|2024-04-07|
+-----------+----------------+--------+----------+
```

In [ ]:
def etl(customers, customers_new):
    # Write code here: return a DataFrame
    return customers

etl(customers, customers_new).show()

## Lesson 2 · Spark SQL inside `etl`

Zillacode's template creates `spark`, so you can answer in SQL whenever that's easier for you:
1. Register each input DataFrame as a temporary view: `orders.createOrReplaceTempView("orders")`.
2. Return `spark.sql("SELECT ...")`, which is a DataFrame like any other.

Everything from Days 12 to 14 has a SQL form: `CASE WHEN ... THEN ... ELSE ... END`, `COALESCE`, `GROUP BY ... HAVING`, every join type (`LEFT ANTI JOIN`, `LEFT SEMI JOIN`, `CROSS JOIN`), window functions (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`), `date_format`, `datediff`... You can also mix the two styles: `F.expr("...")` puts a piece of SQL inside DataFrame code.

Two habits: alias every computed column (`AS revenue`), and filter a window result in an outer query (a `WHERE` can't see a window column of the same `SELECT`). [SQL reference](https://spark.apache.org/docs/latest/sql-ref-syntax.html).

In [ ]:
def etl(orders, products):
    orders.createOrReplaceTempView("orders")
    products.createOrReplaceTempView("products")
    return spark.sql("""
        SELECT p.category, SUM(o.quantity * p.price) AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'delivered'
        GROUP BY p.category
        ORDER BY revenue DESC
    """)

etl(orders, products).show()

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 2850.0|
|      Books|  205.0|
|    Kitchen|   60.0|
+-----------+-------+


In [ ]:
# A window function in SQL, filtered in an outer query
products.createOrReplaceTempView("products")
spark.sql("""
    SELECT category, product_name, price, price_rank
    FROM (SELECT category, product_name, price,
                 RANK() OVER (PARTITION BY category ORDER BY price DESC) AS price_rank
          FROM products)
    WHERE price_rank = 1
    ORDER BY category
""").show()

+-----------+------------+------+----------+
|   category|product_name| price|price_rank|
+-----------+------------+------+----------+
|      Books|    Cookbook|  25.0|         1|
|Electronics|      Laptop|1200.0|         1|
|       Home|   Desk Lamp|  40.0|         1|
|    Kitchen|Coffee Maker|  85.0|         1|
+-----------+------------+------+----------+


In [ ]:
# F.expr: a piece of SQL inside DataFrame code
orders.select(
    "order_id",
    F.expr("CASE WHEN quantity >= 3 THEN 'big' ELSE 'small' END").alias("size"),
    F.expr("date_format(order_date, 'MMM')").alias("month"),
).show(3)

+--------+-----+-----+
|order_id| size|month|
+--------+-----+-----+
|    1001|small|  Jan|
|    1002|  big|  Jan|
|    1003|small|  Jan|
+--------+-----+-----+
only showing top 3 rows


### Exercise

**Q5 · `etl(products)`** Solve it with `spark.sql`: the cheapest product of each category. Show category, product_name and price, sorted by category.

Expected output:

```
+-----------+------------+-----+
|   category|product_name|price|
+-----------+------------+-----+
|      Books|       Novel| 15.0|
|Electronics|  Headphones|150.0|
|       Home|   Desk Lamp| 40.0|
|    Kitchen|     Blender| 60.0|
+-----------+------------+-----+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

## Lesson 3 · Debugging: errors, and the mistakes that give no error

**The errors you will meet**, and what they mean:

| Message (start) | Cause | Fix |
|---|---|---|
| `[UNRESOLVED_COLUMN.WITH_SUGGESTION] ... cannot be resolved. Did you mean ...` | a typo, or a column that an earlier `select`/`groupBy` removed | read the suggested names; `print(df.columns)` |
| `[AMBIGUOUS_REFERENCE]` | both sides of a join have that column | Day 14, Lesson 2 |
| `[CANNOT_CONVERT_COLUMN_INTO_BOOL]` | `and` / `or` / `not` / `if` used on a column | `&`, `\|`, `~`, `F.when` |
| `Column is not iterable` | Python's own `sum`, `max`, `len` used on a column | `F.sum`, `F.max`, `F.length` |
| `'Column' object is not callable` | a DataFrame method called on a column (`F.col("x").count()`) | `F.count("x")` inside `agg` |
| `'NoneType' object has no attribute ...`, or Zillacode's *etl function must return ... not NoneType* | the function returns nothing, or returns `df.show()` (which prints and returns `None`) | `return df` |
| `[DATATYPE_MISMATCH]` | e.g. `F.size` on a text, or `union` of incompatible columns | check the types with `df.printSchema()` |

**Mistakes that give no error, just a wrong answer.** Check these before you press Run:
1. **Columns**: exactly the output schema's names; no helper columns (`rn`, `gap`, `flag`) left over.
2. **Types**: an `avg` or a `/` gives a double; if the schema says Integer, `cast("int")`. Round only when the task says so, and to the stated number of decimals.
3. **Nulls**: filters with `!=` drop null rows; `count("*")` vs `count(col)`; left joins need `coalesce(..., 0)` when the task wants 0.
4. **Joins**: "all X" means a left join; two detail tables joined before aggregating multiply each other (the fan-out trap); null keys never match.
5. **Ties**: `row_number` (exactly N rows) vs `rank` (gaps) vs `dense_rank` (no gaps), and a tie-breaker where the task names one.
6. **Duplicates**: `distinct` before counting days or pages; `count_distinct` when the task says "different" or "unique".
7. **Dates**: text input means `to_date`/`to_timestamp` with the right pattern (`MM` months, `mm` minutes).
8. **Hidden test**: no values copied from the example (ids, dates, counts).

In [ ]:
# A typo: the message suggests the real names
try:
    orders.select("order_dte")
except Exception as e:
    print(type(e).__name__, "->", str(e).split("SQLSTATE")[0][:230])

AnalysisException -> [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `order_dte` cannot be resolved. Did you mean one of the following? [`order_date`, `order_id`, `product_id`, `quantity`, `status`].


In [ ]:
# show() prints and returns None: never assign or return it
result = orders.filter(F.col("quantity") > 3).show()
print("result is", result)

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1009|          3|       105|       4|2024-03-05|delivered|
|    1014|          5|       105|       5|2024-03-20|delivered|
+--------+-----------+----------+--------+----------+---------+

result is None


In [ ]:
# Python's own functions on a column, and a DataFrame method on a column
for attempt in [lambda: sum(F.col("quantity")), lambda: F.col("quantity").count()]:
    try:
        attempt()
    except Exception as e:
        print(type(e).__name__, "->", str(e).split(".")[0][:120])

PySparkTypeError -> [NOT_ITERABLE] Column is not iterable
TypeError -> 'Column' object is not callable


## Lesson 4 · UDFs: a Python function as a column function

`F.udf(python_function, return_type)` turns a plain Python function into a column function. Some official Zillacode solutions use one (#31 checks weekends with Python's `datetime`, #40 counts list items with Python's `split`). It works, but:
* A UDF runs Python for every row, outside Spark's optimizer, so it's slower than built-in functions (theory Day 8).
* **The function itself must handle `None`**: Spark passes null as `None`, and `None.split(",")` crashes the job.
* Almost every case has a built-in: `F.dayofweek(F.to_date(col, "MM/dd/yyyy")).isin(1, 7)` for weekends, `F.size(F.split(...))` with a null guard for counting.

Use a UDF only when no built-in function (or combination of them) can do the job. [udf docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html).

In [ ]:
import datetime                                   # Zillacode's template imports the module like this
from pyspark.sql.types import IntegerType, BooleanType

count_items = F.udf(lambda s: len(s.split(",")) if s else 0, IntegerType())        # handles None and ""
is_weekend = F.udf(lambda s: datetime.datetime.strptime(s, "%m/%d/%Y").weekday() >= 5 if s else None, BooleanType())

t = spark.createDataFrame([("a,b,c", "03/02/2024"), ("", "03/04/2024"), (None, None)], "items string, day string")
t.select(
    "items", "day",
    count_items("items").alias("count_udf"),
    F.when(F.col("items").isNull() | (F.col("items") == ""), 0).otherwise(F.size(F.split("items", ","))).alias("count_builtin"),
    is_weekend("day").alias("weekend_udf"),
    F.dayofweek(F.to_date("day", "MM/dd/yyyy")).isin(1, 7).alias("weekend_builtin"),
).show()

+-----+----------+---------+-------------+-----------+---------------+
|items|       day|count_udf|count_builtin|weekend_udf|weekend_builtin|
+-----+----------+---------+-------------+-----------+---------------+
|a,b,c|03/02/2024|        3|            3|       true|           true|
|     |03/04/2024|        0|            0|      false|          false|
| NULL|      NULL|        0|            0|       NULL|           NULL|
+-----+----------+---------+-------------+-----------+---------------+


## Mock exam: 8 problems in 90 minutes

Set a timer for 90 minutes. No hints and no looking back at the lessons: write each `etl` from scratch, as on Zillacode. Suggested time: easy 5 minutes, medium 10, hard 15; what's left is your buffer. Check your answers only when the time is up.

**New data, built like Zillacode builds it.** A bike-sharing service in Uzbekistan. The cell below creates the tables from lists of Python dicts with `spark.createDataFrame`, exactly as Zillacode does, so:
* the columns are in **alphabetical order**,
* whole numbers are `bigint`, and decimals are `double`,
* **times and dates are text**.

| Table | Columns |
|---|---|
| **stations** | *station_id*, *station_name*, *city* |
| **members** | *member_id*, *full_name*, *plan* (`annual`, `monthly` or `trial`), *joined* (text `MM/dd/yyyy`, or `unknown`) |
| **rides** | *ride_id*, *member_id*, *bike_id*, *start_station*, *end_station* (station ids), *start_time*, *end_time* (text `yyyy-MM-dd HH:mm:ss`), *price* (dollars) |

In [ ]:
stations = spark.createDataFrame([
    {"station_id": 1, "station_name": "Amir Temur Square", "city": "Tashkent"},
    {"station_id": 2, "station_name": "Chorsu Bazaar", "city": "Tashkent"},
    {"station_id": 3, "station_name": "Registan", "city": "Samarkand"},
    {"station_id": 4, "station_name": "Siab Bazaar", "city": "Samarkand"},
    {"station_id": 5, "station_name": "Lyabi-Hauz", "city": "Bukhara"},
])
members = spark.createDataFrame([
    {"member_id": 1, "full_name": "Nodira Yusupova", "plan": "annual", "joined": "01/15/2024"},
    {"member_id": 2, "full_name": "Timur Saidov", "plan": "monthly", "joined": "02/03/2024"},
    {"member_id": 3, "full_name": "Laylo Karimova", "plan": "annual", "joined": "02/17/2024"},
    {"member_id": 4, "full_name": "Javlon Rashidov", "plan": "monthly", "joined": "03/02/2024"},
    {"member_id": 5, "full_name": "Malika Tursunova", "plan": "trial", "joined": "unknown"},
])
rides = spark.createDataFrame([
    {"ride_id": 1, "member_id": 1, "bike_id": "B1", "start_station": 1, "end_station": 2, "start_time": "2024-03-01 08:00:00", "end_time": "2024-03-01 08:25:00", "price": 1.5},
    {"ride_id": 2, "member_id": 2, "bike_id": "B2", "start_station": 3, "end_station": 4, "start_time": "2024-03-01 09:00:00", "end_time": "2024-03-01 09:20:00", "price": 1.2},
    {"ride_id": 3, "member_id": 1, "bike_id": "B1", "start_station": 2, "end_station": 1, "start_time": "2024-03-01 18:10:00", "end_time": "2024-03-01 18:50:00", "price": 2.4},
    {"ride_id": 4, "member_id": 1, "bike_id": "B1", "start_station": 1, "end_station": 1, "start_time": "2024-03-02 07:45:00", "end_time": "2024-03-02 08:30:00", "price": 2.7},
    {"ride_id": 5, "member_id": 3, "bike_id": "B2", "start_station": 3, "end_station": 3, "start_time": "2024-03-02 10:00:00", "end_time": "2024-03-02 10:35:00", "price": 2.1},
    {"ride_id": 6, "member_id": 2, "bike_id": "B3", "start_station": 5, "end_station": 5, "start_time": "2024-03-02 16:00:00", "end_time": "2024-03-02 16:15:00", "price": 0.9},
    {"ride_id": 7, "member_id": 1, "bike_id": "B1", "start_station": 2, "end_station": 2, "start_time": "2024-03-03 08:05:00", "end_time": "2024-03-03 08:20:00", "price": 0.9},
    {"ride_id": 8, "member_id": 3, "bike_id": "B2", "start_station": 3, "end_station": 4, "start_time": "2024-03-03 11:00:00", "end_time": "2024-03-03 11:50:00", "price": 3.0},
    {"ride_id": 9, "member_id": 2, "bike_id": "B3", "start_station": 5, "end_station": 6, "start_time": "2024-03-04 12:00:00", "end_time": "2024-03-04 12:30:00", "price": 1.8},
    {"ride_id": 10, "member_id": 3, "bike_id": "B1", "start_station": 2, "end_station": 1, "start_time": "2024-03-04 19:00:00", "end_time": "2024-03-04 19:10:00", "price": 0.6},
    {"ride_id": 11, "member_id": 1, "bike_id": "B2", "start_station": 4, "end_station": 3, "start_time": "2024-03-05 09:00:00", "end_time": "2024-03-05 09:40:00", "price": 2.4},
    {"ride_id": 12, "member_id": 4, "bike_id": "B3", "start_station": 1, "end_station": 2, "start_time": "2024-03-06 08:00:00", "end_time": "2024-03-06 08:35:00", "price": 2.1},
    {"ride_id": 13, "member_id": 3, "bike_id": "B3", "start_station": 1, "end_station": 1, "start_time": "2024-03-05 17:00:00", "end_time": "2024-03-05 17:20:00", "price": 1.2},
])
rides.printSchema()
stations.show()
members.show()
rides.show(truncate=False)

root
 |-- bike_id: string (nullable = true)
 |-- end_station: long (nullable = true)
 |-- end_time: string (nullable = true)
 |-- member_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- ride_id: long (nullable = true)
 |-- start_station: long (nullable = true)
 |-- start_time: string (nullable = true)

+---------+----------+-----------------+
|     city|station_id|     station_name|
+---------+----------+-----------------+
| Tashkent|         1|Amir Temur Square|
| Tashkent|         2|    Chorsu Bazaar|
|Samarkand|         3|         Registan|
|Samarkand|         4|      Siab Bazaar|
|  Bukhara|         5|       Lyabi-Hauz|
+---------+----------+-----------------+

+----------------+----------+---------+-------+
|       full_name|    joined|member_id|   plan|
+----------------+----------+---------+-------+
| Nodira Yusupova|01/15/2024|        1| annual|
|    Timur Saidov|02/03/2024|        2|monthly|
|  Laylo Karimova|02/17/2024|        3| annual|
| Javlon Rashidov|

**Q6 · `etl(rides)`** (easy) Which rides lasted more than 30 minutes? Show ride_id, member_id and minutes (a whole number), sorted by ride_id.

Expected output:

```
+-------+---------+-------+
|ride_id|member_id|minutes|
+-------+---------+-------+
|      3|        1|     40|
|      4|        1|     45|
|      5|        3|     35|
|      8|        3|     50|
|     11|        1|     40|
|     12|        4|     35|
+-------+---------+-------+
```

In [ ]:
def etl(rides):
    # Write code here: return a DataFrame
    return rides

etl(rides).show()

**Q7 · `etl(members)`** (easy) Show member_id, first_name, last_name, plan (in upper case) and joined_date (a date; null when the text isn't a date), sorted by member_id.

Expected output:

```
+---------+----------+---------+-------+-----------+
|member_id|first_name|last_name|   plan|joined_date|
+---------+----------+---------+-------+-----------+
|        1|    Nodira| Yusupova| ANNUAL| 2024-01-15|
|        2|     Timur|   Saidov|MONTHLY| 2024-02-03|
|        3|     Laylo| Karimova| ANNUAL| 2024-02-17|
|        4|    Javlon| Rashidov|MONTHLY| 2024-03-02|
|        5|    Malika|Tursunova|  TRIAL|       NULL|
+---------+----------+---------+-------+-----------+
```

In [ ]:
def etl(members):
    # Write code here: return a DataFrame
    return members

etl(members).show()

**Q8 · `etl(members, rides)`** (easy) Which members have never taken a ride? Show member_id and full_name.

Expected output:

```
+---------+----------------+
|member_id|       full_name|
+---------+----------------+
|        5|Malika Tursunova|
+---------+----------------+
```

In [ ]:
def etl(members, rides):
    # Write code here: return a DataFrame
    return members

etl(members, rides).show()

**Q9 · `etl(rides, stations)`** (medium) For each city, count the rides that started there and add up their prices: city, rides, revenue (2 decimals). Highest revenue first.

Expected output:

```
+---------+-----+-------+
|     city|rides|revenue|
+---------+-----+-------+
| Tashkent|    7|   11.4|
|Samarkand|    4|    8.7|
|  Bukhara|    2|    2.7|
+---------+-----+-------+
```

In [ ]:
def etl(rides, stations):
    # Write code here: return a DataFrame
    return rides

etl(rides, stations).show()

**Q10 · `etl(rides, stations)`** (medium) Show every ride with the names of its stations: ride_id, from_station and to_station (empty when the station is unknown), sorted by ride_id.

Expected output:

```
+-------+-----------------+-----------------+
|ride_id|     from_station|       to_station|
+-------+-----------------+-----------------+
|      1|Amir Temur Square|    Chorsu Bazaar|
|      2|         Registan|      Siab Bazaar|
|      3|    Chorsu Bazaar|Amir Temur Square|
|      4|Amir Temur Square|Amir Temur Square|
|      5|         Registan|         Registan|
|      6|       Lyabi-Hauz|       Lyabi-Hauz|
|      7|    Chorsu Bazaar|    Chorsu Bazaar|
|      8|         Registan|      Siab Bazaar|
|      9|       Lyabi-Hauz|             NULL|
|     10|    Chorsu Bazaar|Amir Temur Square|
|     11|      Siab Bazaar|         Registan|
|     12|Amir Temur Square|    Chorsu Bazaar|
|     13|Amir Temur Square|Amir Temur Square|
+-------+-----------------+-----------------+
```

In [ ]:
def etl(rides, stations):
    # Write code here: return a DataFrame
    return rides

etl(rides, stations).show()

**Q11 · `etl(rides, stations)`** (medium) Find each member's favourite start station: the one they started the most rides from (on a tie, the smaller station_id). Show member_id, station_name and rides, sorted by member_id.

Expected output:

```
+---------+-----------------+-----+
|member_id|     station_name|rides|
+---------+-----------------+-----+
|        1|Amir Temur Square|    2|
|        2|       Lyabi-Hauz|    2|
|        3|         Registan|    2|
|        4|Amir Temur Square|    1|
+---------+-----------------+-----+
```

In [ ]:
def etl(rides, stations):
    # Write code here: return a DataFrame
    return rides

etl(rides, stations).show()

**Q12 · `etl(rides)`** (hard) A bike should start each ride where its previous ride ended. Find the rides where it didn't (staff moved the bike): bike_id, ride_id, expected_station (where the previous ride ended) and actual_station (where this ride started). Sort by bike_id, then ride_id.

Expected output:

```
+-------+-------+----------------+--------------+
|bike_id|ride_id|expected_station|actual_station|
+-------+-------+----------------+--------------+
|     B1|      7|               1|             2|
|     B2|      5|               4|             3|
|     B3|     13|               6|             1|
+-------+-------+----------------+--------------+
```

In [ ]:
def etl(rides):
    # Write code here: return a DataFrame
    return rides

etl(rides).show()

**Q13 · `etl(rides)`** (hard) What is each member's longest streak of consecutive days with at least one ride? Show member_id and longest_streak, sorted by member_id.

Expected output:

```
+---------+--------------+
|member_id|longest_streak|
+---------+--------------+
|        1|             3|
|        2|             2|
|        3|             4|
|        4|             1|
+---------+--------------+
```

In [ ]:
def etl(rides):
    # Write code here: return a DataFrame
    return rides

etl(rides).show()

## Next: Zillacode

You now know every tool its 52 PySpark problems use. Set it up (the last task of Day 15 in the roadmap), then solve the problems Easy first, then Medium, then Hard. For each one: read the output schema, plan the steps in a comment, write `etl`, and check the "no error, wrong answer" list from Lesson 3 before you press Run. If a problem fails, compare your output with the expected one row by row; the difference almost always points at nulls, ties, types or an extra column.

Zillacode's problems by topic (number and level: E, M, H), so you can drill a weak topic:

| Topic | Problems |
|---|---|
| Complex Joins | 2M, 3H, 5E, 10M, 13H, 14E, 15H, 18M, 20H, 23M, 24H, 27M, 31M, 38E, 46E, 47E, 49M, 52E |
| Aggregate Functions | 3H, 5E, 8E, 10M, 15H, 19M, 20H, 23M, 25E, 27M, 32M, 35E, 36E, 39E, 42E, 44M |
| Windows Functions | 7H, 9M, 13H, 15H, 21H, 22H, 24H, 28M, 29H, 30H, 37H, 43H, 45H |
| Regular Expressions | 4M, 6E, 17E, 26H, 30H, 51H |
| Conditional Logic | 1E, 7H, 32M, 36E, 50E |
| Datetime Operations | 1E, 20H, 22H, 30H, 31M |
| Case When Statements | 16E, 21H, 33M, 41E, 45H |
| String Manipulation | 2M, 21H, 40M, 48M |
| Null Handling | 11E, 14E, 26H |
| Mathematical Functions | 29H, 45H, 51H |
| Distinct Function | 8E, 13H |
| Union Statements | 12E, 34E |
| Pivot Function | 3H |

## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than ...</summary>

```python
def etl(events):
    w = W.partitionBy("customer_id").orderBy("event_time")
    gap_minutes = (F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60
    return (events.dropDuplicates()
                  .filter(F.col("customer_id").isin(1, 2))
                  .withColumn("new_session", F.when(gap_minutes.isNull() | (gap_minutes > 30), 1).otherwise(0))
                  .withColumn("session", F.sum("new_session").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)))
                  .select("customer_id", "event_time", "page", "session")
                  .orderBy("customer_id", "event_time"))

etl(events).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;What is each customer's longest streak of consecutive days with at least one event? Show customer_id and ...</summary>

```python
def etl(events):
    days = events.select("customer_id", F.to_date("event_time").alias("day")).distinct()
    w = W.partitionBy("customer_id").orderBy("day")
    streaks = (days.withColumn("anchor", F.date_sub("day", F.row_number().over(w)))
                   .groupBy("customer_id", "anchor")
                   .agg(F.count("*").alias("days")))
    return (streaks.groupBy("customer_id")
                   .agg(F.max("days").alias("longest_streak"))
                   .orderBy("customer_id"))

etl(events).show()
```

</details>

<details>
<summary><b>Q3</b> &nbsp;Find the customers whose first order (by date, then order_id) was delivered but whose latest order was not. ...</summary>

```python
def etl(orders):
    by_time = F.struct("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.min_by("status", by_time).alias("first_status"), F.max_by("status", by_time).alias("last_status"))
                  .filter((F.col("first_status") == "delivered") & (F.col("last_status") != "delivered"))
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q4</b> &nbsp;Build the current customer list from two snapshots: customers (taken on 2024-03-31) and customers_new (taken ...</summary>

```python
def etl(customers, customers_new):
    both = (customers.withColumn("snapshot", F.to_date(F.lit("2024-03-31")))
                     .unionByName(customers_new.withColumn("snapshot", F.to_date(F.lit("2024-04-07")))))
    w = W.partitionBy("customer_id").orderBy(F.desc("snapshot"))
    return (both.withColumn("rn", F.row_number().over(w))
                .filter(F.col("rn") == 1)
                .select("customer_id", "name", "city", "snapshot")
                .orderBy("customer_id"))

etl(customers, customers_new).show()
```

</details>

<details>
<summary><b>Q5</b> &nbsp;Solve it with spark.sql: the cheapest product of each category. Show category, product_name and price, sorted ...</summary>

```python
def etl(products):
    products.createOrReplaceTempView("products")
    return spark.sql("""
        SELECT category, product_name, price
        FROM (SELECT category, product_name, price,
                     ROW_NUMBER() OVER (PARTITION BY category ORDER BY price) AS rn
              FROM products)
        WHERE rn = 1
        ORDER BY category
    """)

etl(products).show()
```

</details>

<details>
<summary><b>Q6</b> &nbsp;(easy) Which rides lasted more than 30 minutes? Show ride_id, member_id and minutes (a whole number), sorted ...</summary>

```python
def etl(rides):
    minutes = ((F.unix_timestamp(F.to_timestamp("end_time")) - F.unix_timestamp(F.to_timestamp("start_time"))) / 60).cast("int")
    return (rides.select("ride_id", "member_id", minutes.alias("minutes"))
                 .filter(F.col("minutes") > 30)
                 .orderBy("ride_id"))

etl(rides).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;(easy) Show member_id, first_name, last_name, plan (in upper case) and joined_date (a date; null when the ...</summary>

```python
def etl(members):
    parts = F.split("full_name", " ")
    return (members.select("member_id", parts[0].alias("first_name"), parts[1].alias("last_name"),
                           F.upper("plan").alias("plan"), F.to_date("joined", "MM/dd/yyyy").alias("joined_date"))
                   .orderBy("member_id"))

etl(members).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;(easy) Which members have never taken a ride? Show member_id and full_name.</summary>

```python
def etl(members, rides):
    return members.join(rides, "member_id", "left_anti").select("member_id", "full_name").orderBy("member_id")

etl(members, rides).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;(medium) For each city, count the rides that started there and add up their prices: city, rides, revenue (2 ...</summary>

```python
def etl(rides, stations):
    return (rides.join(stations, F.col("start_station") == F.col("station_id"))
                 .groupBy("city")
                 .agg(F.count("*").alias("rides"), F.round(F.sum("price"), 2).alias("revenue"))
                 .orderBy(F.desc("revenue")))

etl(rides, stations).show()
```

</details>

<details>
<summary><b>Q10</b> &nbsp;(medium) Show every ride with the names of its stations: ride_id, from_station and to_station (empty when the ...</summary>

```python
def etl(rides, stations):
    start = stations.select(F.col("station_id").alias("start_station"), F.col("station_name").alias("from_station"))
    end = stations.select(F.col("station_id").alias("end_station"), F.col("station_name").alias("to_station"))
    return (rides.join(start, "start_station", "left")
                 .join(end, "end_station", "left")
                 .select("ride_id", "from_station", "to_station")
                 .orderBy("ride_id"))

etl(rides, stations).show()
```

</details>

<details>
<summary><b>Q11</b> &nbsp;(medium) Find each member's favourite start station: the one they started the most rides from (on a tie, the ...</summary>

```python
def etl(rides, stations):
    counts = rides.groupBy("member_id", "start_station").agg(F.count("*").alias("rides"))
    w = W.partitionBy("member_id").orderBy(F.desc("rides"), "start_station")
    return (counts.withColumn("rn", F.row_number().over(w))
                  .filter(F.col("rn") == 1)
                  .join(stations, F.col("start_station") == F.col("station_id"))
                  .select("member_id", "station_name", "rides")
                  .orderBy("member_id"))

etl(rides, stations).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;(hard) A bike should start each ride where its previous ride ended. Find the rides where it didn't (staff ...</summary>

```python
def etl(rides):
    w = W.partitionBy("bike_id").orderBy(F.to_timestamp("start_time"))
    return (rides.withColumn("expected_station", F.lag("end_station").over(w))
                 .filter(F.col("expected_station") != F.col("start_station"))
                 .select("bike_id", "ride_id", "expected_station", F.col("start_station").alias("actual_station"))
                 .orderBy("bike_id", "ride_id"))

etl(rides).show()
```

Order by the start time, not by ride_id: ride 13 started before ride 12, and ordering bike B3's rides by id gives a wrong answer. The first ride of each bike has a null `expected_station`, and `null != x` is null, so the filter drops it without an extra test.

</details>

<details>
<summary><b>Q13</b> &nbsp;(hard) What is each member's longest streak of consecutive days with at least one ride? Show member_id and ...</summary>

```python
def etl(rides):
    days = rides.select("member_id", F.to_date("start_time").alias("day")).distinct()
    w = W.partitionBy("member_id").orderBy("day")
    return (days.withColumn("anchor", F.date_sub("day", F.row_number().over(w)))
                .groupBy("member_id", "anchor")
                .agg(F.count("*").alias("days"))
                .groupBy("member_id")
                .agg(F.max("days").alias("longest_streak"))
                .orderBy("member_id"))

etl(rides).show()
```

</details>